In [1]:
# ============================================================================
# DATA LOADING 
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 100)

# Load the optimised features CSV (all 257,673 samples, 6 PCs + target)
dftotal = pd.read_csv('/home/e-ackerson/GIT_hub/Research_Project/DL_models/CNN_models_evals/16_optimised_6/16_optimal_features_ids')

# Drop the unnamed index column if present
if 'Unnamed: 0' in dftotal.columns:
    dftotal = dftotal.drop(columns=['Unnamed: 0'])

DFtotal = dftotal[[
    'Fused_proto_PC3', 'Fused_dur_PC2', 'Fused_proto_PC1',
    'Fused_proto_PC9', 'Fused_proto_PC6', 'Fused_dur_PC1',
    'attack_cat'
]]

print("DFtotal shape:", DFtotal.shape)
print("Unique original classes:", sorted(DFtotal['attack_cat'].unique()))

# ============================================================================
# APPLY MACRO-CLASS MAPPING
# ============================================================================
# Original 10 classes (alphabetical integer encoding):
#   0 Analysis, 1 Backdoor, 2 DoS, 3 Exploits, 4 Fuzzers,
#   5 Generic, 6 Normal, 7 Reconnaissance, 8 Shellcode, 9 Worms

category_map = {
    2: 'DoS-Fuzzers',
    4: 'DoS-Fuzzers',
    1: 'System-Exploits',
    3: 'System-Exploits',
    8: 'System-Exploits',
    9: 'Worm-Recon',
    0: 'Worm-Recon',
    7: 'Worm-Recon',
    5: 'Generic',
    6: 'Normal'
}

DFtotal['attack_cat'] = DFtotal['attack_cat'].map(category_map)

print("\nMacro-class distribution:")
print(DFtotal['attack_cat'].value_counts().sort_index())

# Re-encode macro-classes to contiguous integers (0-4)
from sklearn.preprocessing import LabelEncoder

macro_encoder = LabelEncoder()
DFtotal['attack_cat'] = macro_encoder.fit_transform(DFtotal['attack_cat'])

print("\nMacro-class integer mapping:")
for i, name in enumerate(macro_encoder.classes_):
    print(f"  {i} -> {name}")


print("Macro-class distribution:")
print(DFtotal['attack_cat'].value_counts().sort_index())
# ============================================================================
# TRAIN / TEST SPLIT
# ============================================================================
from sklearn.model_selection import train_test_split

y = DFtotal['attack_cat']
x = DFtotal.drop(columns={'attack_cat'})

xtrain_val, xtest, ytrain_val, ytest = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain:", xtrain_val.shape, "Test:", xtest.shape)


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
FEATURES = [
    'Fused_proto_PC3', 'Fused_dur_PC2', 'Fused_proto_PC1',
    'Fused_proto_PC9', 'Fused_proto_PC6', 'Fused_dur_PC1',
]

x_train = xtrain_val[FEATURES].copy()
x_test  = xtest[FEATURES].copy()

print("x_train:", x_train.shape, "x_test:", x_test.shape)


# ============================================================================
# CLASS WEIGHTS
# ============================================================================
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(ytrain_val)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=ytrain_val
)
class_weights = dict(zip(classes, class_weights_array))
print("Class weights:", class_weights)


# ============================================================================
# RESHAPE FOR 1D-CNN
# ============================================================================
y_train = ytrain_val.values
y_test  = ytest.values
num_classes = len(np.unique(y_train))

n_features = x_train.shape[1]

X_train_cnn = x_train.values.reshape(-1, n_features, 1)
X_test_cnn  = x_test.values.reshape(-1, n_features, 1)

print("X_train_cnn:", X_train_cnn.shape)
print("X_test_cnn: ", X_test_cnn.shape)
print("num_classes:", num_classes)
print("n_features: ", n_features)


#...........................................................................
#............................... ARTIFACTS ..................................
import os
import joblib
import json

# Create the artifacts directory if it does not exist
os.makedirs('artifacts_designed_16', exist_ok=True)
print("Artifacts directory ready: artifacts_designed_16/")

# ............................................................................
# Saving Macro-class label encoder
# ............................................................................
joblib.dump(macro_encoder, 'artifacts_designed_16/macro_label_encoder.pkl')
print("\nSaved label encoder. Classes:")
for i, name in enumerate(macro_encoder.classes_):
    print(f"  {i} -> {name}")

label_map = {int(i): name for i, name in enumerate(macro_encoder.classes_)}
with open('artifacts_designed_16/label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)
print("Saved label_map.json")

# ............................................................................
# Save feature order
# ............................................................................
FEATURE_ORDER = [
    'Fused_proto_PC3', 'Fused_dur_PC2', 'Fused_proto_PC1',
    'Fused_proto_PC9', 'Fused_proto_PC6', 'Fused_dur_PC1',
]
with open('artifacts_designed_16/feature_order.json', 'w') as f:
    json.dump(FEATURE_ORDER, f, indent=2)
print("\nSaved feature_order.json:", FEATURE_ORDER)

# ............................................................................
# Verify what was saved
# ............................................................................
print("\nArtifacts directory contents:")
for fname in os.listdir('artifacts_designed_16'):
    fpath = os.path.join('artifacts_designed_16', fname)
    size = os.path.getsize(fpath)
    print(f"  {fname}  ({size} bytes)")

DFtotal shape: (257673, 7)
Unique original classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Macro-class distribution:
attack_cat
DoS-Fuzzers        40599
Generic            58871
Normal             93000
System-Exploits    48365
Worm-Recon         16838
Name: count, dtype: int64

Macro-class integer mapping:
  0 -> DoS-Fuzzers
  1 -> Generic
  2 -> Normal
  3 -> System-Exploits
  4 -> Worm-Recon
Macro-class distribution:
attack_cat
0    40599
1    58871
2    93000
3    48365
4    16838
Name: count, dtype: int64

Train: (206138, 6) Test: (51535, 6)
x_train: (206138, 6) x_test: (51535, 6)
Class weights: {0: 1.269361741432926, 1: 0.8753763509353037, 2: 0.5541344086021506, 3: 1.0655329267031945, 4: 3.0606978470675577}
X_train_cnn: (206138, 6, 1)
X_test_cnn:  (51535, 6, 1)
num_classes: 5
n_features:  6
Artifacts directory ready: artifacts_designed_16/

Saved label encoder. Classes:
  0 -> DoS-Fuzzers
  1 -> Generic
  2 -> Normal
  3 -> System-Exploits
  4 -> Worm-Recon
Saved label_map.json

Saved f

In [3]:
import os

print("\ndesigned_16 artifacts:")
for f in sorted(os.listdir('artifacts_designed_16')):
    path = os.path.join('artifacts_designed_16', f)
    print(f"  {f}  ({os.path.getsize(path)} bytes)")


designed_16 artifacts:
  cnn_designed_16_best.h5  (797984 bytes)
  feature_order.json  (124 bytes)
  label_map.json  (106 bytes)
  macro_label_encoder.pkl  (530 bytes)
